# Video Face Swap Demo - 2 người (inswapper + InsightFace)

**Logic:** 1 video mẫu có 2 người (ví dụ cảnh hôn nhau) + 2 ảnh khuôn mặt (người A, người B) → video mới giữ nguyên chuyển động/nền gốc, mỗi người trong video được thay đúng bằng khuôn mặt tương ứng đã cung cấp.

**Công nghệ dùng:**
- `insightface` (buffalo_l) để detect + align khuôn mặt từng frame
- `inswapper_128.onnx` để swap khuôn mặt
- `GFPGAN` (tùy chọn) để làm nét/phục hồi mặt sau khi swap
- `ffmpeg` để tách/ghép audio và dựng lại video

**Trước khi chạy:** Runtime > Change runtime type > chọn GPU (T4).

⚠️ Lưu ý: công nghệ face-swap có thể bị dùng sai mục đích (deepfake giả mạo người khác mà không có sự đồng ý). Chỉ dùng với ảnh/video của chính bạn hoặc người đã đồng ý, và cân nhắc gắn watermark/disclosure khi xuất bản sản phẩm thật.

## 0. ~~Cài môi trường Python 3.10 qua Miniconda~~ — HIỆN KHÔNG DÙNG ĐƯỢC

⚠️ **Cập nhật:** cách này hiện KHÔNG chạy được, vì bản thân package `condacolab` tự kiểm tra và chỉ chấp nhận Colab đang chạy đúng Python 3.12 (`assert colab_python == '3.12'` viết cứng trong code), trong khi Colab hiện đã lên Python 3.13 — `condacolab` chưa kịp cập nhật theo. Đây là giới hạn của chính tool `condacolab`, không phải do cấu hình sai.

**→ Bỏ qua toàn bộ mục 0, chuyển thẳng xuống mục 1.** Các cell ở mục 1 trở đi đã có sẵn patch để chạy được trên Python 3.13 của Colab hiện tại.

In [ ]:
# Log phiên bản Python Colab đang cấp (để biết đang chạy bản nào)
import sys, platform
print('Python version:', sys.version)
print('Platform:', platform.platform())

## 1. Cài đặt thư viện

In [ ]:
import sys
print('Python đang chạy trong cell này:', sys.version)

In [ ]:
# Ghim setuptools < 82: bản setuptools mới (>=82) đã bỏ hẳn module 'distutils',
# trong khi basicsr (dependency của GFPGAN) và torch trên Colab vẫn cần distutils/setuptools cũ.
!pip install -q "setuptools==79.0.1" wheel
!pip install -q cython numpy

# --no-build-isolation: để insightface dùng đúng cython/numpy vừa cài ở trên,
# thay vì pip tự tạo môi trường tạm cô lập (không thấy cython) rồi build lỗi 'egg_info'.
!pip install -q --no-build-isolation insightface==0.7.3

# KHÔNG cài opencv-python-headless: Colab đã có sẵn opencv-python, mà hai package này
# dùng chung namespace `cv2` -> cài đè lên nhau hay để lại .so lẫn lộn gây lỗi khó hiểu.
# (insightface cũng khai báo opencv-python là dependency nên chắc chắn có cv2.)
!pip install -q onnxruntime-gpu gfpgan facexlib
!apt-get -qq install -y ffmpeg > /dev/null

### Fix riêng cho `basicsr` (bug với Python bản mới trên Colab)

`basicsr` có bug trong `setup.py`: dùng `exec(...)` rồi đọc `locals()['__version__']` để lấy version. Ở Python 3.13, `exec()` trong 1 hàm không còn ghi ngược lại `locals()` đáng tin cậy (thay đổi theo PEP 667) → lỗi `KeyError: '__version__'`. Cell dưới tải source về, patch đúng chỗ này (`locals()` → `globals()`), rồi cài từ bản đã sửa.

In [ ]:
import subprocess, sys, os, tarfile, urllib.request, json

os.makedirs('/tmp/basicsr_src', exist_ok=True)

# Tải trực tiếp từ PyPI bằng urllib (KHÔNG dùng `pip download`, vì pip cũng phải chạy
# setup.py egg_info để lấy metadata -> dính đúng bug KeyError trước khi kịp patch).
with urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json') as resp:
    pkg_info = json.load(resp)

sdist_url = None
for url_info in pkg_info['urls']:
    if url_info['packagetype'] == 'sdist':
        sdist_url = url_info['url']
        break
assert sdist_url is not None, 'Không tìm thấy sdist của basicsr trên PyPI.'

tar_name = sdist_url.split('/')[-1]
tar_path = f'/tmp/basicsr_src/{tar_name}'
urllib.request.urlretrieve(sdist_url, tar_path)
print(f'Đã tải: {tar_name}')

extract_dir = '/tmp/basicsr_build'
os.makedirs(extract_dir, exist_ok=True)
with tarfile.open(tar_path) as tar:
    # Lấy tên thư mục gốc từ chính nội dung tar thay vì suy ra bằng tar_name.replace('.tar.gz', ''),
    # vì sdist trên PyPI không bắt buộc phải là .tar.gz.
    root_names = {m.name.split('/')[0] for m in tar.getmembers() if m.name.strip('./')}
    assert len(root_names) == 1, f'sdist có cấu trúc thư mục lạ: {root_names}'
    root_name = root_names.pop()
    # filter='data': Python 3.12+ deprecate extractall không có filter, 3.14 đổi default.
    tar.extractall(extract_dir, filter='data')

pkg_dir = os.path.join(extract_dir, root_name)
setup_py_path = os.path.join(pkg_dir, 'setup.py')
assert os.path.exists(setup_py_path), f'Không thấy setup.py trong {pkg_dir}'

with open(setup_py_path, 'r') as f:
    content = f.read()

# Fix bug Python 3.13: exec() trong hàm không ghi ngược locals() đáng tin cậy
content = content.replace(
    "exec(compile(f.read(), version_file, 'exec'))",
    "exec(compile(f.read(), version_file, 'exec'), globals())"
)
content = content.replace(
    "return locals()['__version__']",
    "return globals()['__version__']"
)

with open(setup_py_path, 'w') as f:
    f.write(content)

print('Đã patch setup.py xong, tiến hành cài basicsr từ source đã sửa...')
# sys.executable -m pip: đảm bảo cài vào đúng interpreter đang chạy notebook,
# thay vì lệnh `pip` bất kỳ đứng đầu PATH.
install_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', pkg_dir],
    capture_output=True, text=True
)
print(install_result.stdout[-3000:])
print(install_result.stderr[-3000:])
if install_result.returncode != 0:
    raise RuntimeError('Cài basicsr thất bại, xem log lỗi ở trên để biết nguyên nhân cụ thể.')
print('Cài basicsr thành công.')

## 2. Tải model (inswapper + buffalo_l + GFPGAN)

Model `inswapper_128.onnx` không được host chính thức trên GitHub release nữa do vấn đề chính sách, 
nên bạn cần tự tải và upload lên Google Drive của mình, hoặc dùng link mirror cộng đồng (huggingface). 
Cell dưới thử tải từ 1 mirror phổ biến trên Hugging Face — nếu lỗi, bạn tải thủ công rồi upload vào `/content/`.

In [ ]:
import os
os.makedirs('/content/models', exist_ok=True)

INSWAPPER_PATH = '/content/models/inswapper_128.onnx'
GFPGAN_PATH = '/content/models/GFPGANv1.4.pth'

# Mirror cộng đồng trên Hugging Face (kiểm tra lại link còn sống trước khi chạy)
!wget -q -O {INSWAPPER_PATH} https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx

# GFPGAN weights cho face restoration (tùy chọn)
!wget -q -O {GFPGAN_PATH} https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth

# wget vẫn coi là "thành công" khi server trả 404 / trang HTML -> phải tự kiểm tra dung lượng.
# Nếu không, mãi tới cell load model mới nổ với lỗi onnx/torch rất khó đoán nguyên nhân.
def check_download(path, min_mb, hint):
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    assert size_mb >= min_mb, (
        f'Tải {os.path.basename(path)} thất bại (chỉ {size_mb:.1f} MB, cần >= {min_mb} MB). {hint}'
    )
    print(f'OK  {os.path.basename(path)}: {size_mb:.1f} MB')

check_download(INSWAPPER_PATH, 200,
               'Link mirror có thể đã chết -> tự tải rồi upload vào /content/models/.')
check_download(GFPGAN_PATH, 300,
               'Kiểm tra lại link GitHub release của GFPGAN.')

!ls -lh /content/models/

## 3. Upload 2 ảnh khuôn mặt (người A, người B) + video mẫu (2 người)

In [ ]:
from google.colab import files

def pick_one(uploaded, what):
    names = list(uploaded.keys())
    assert len(names) > 0, f'Chưa upload {what} (bấm Cancel?). Chạy lại cell này.'
    if len(names) > 1:
        print(f'  (đã upload {len(names)} file, dùng file đầu tiên: {names[0]})')
    return names[0]

print('>> Upload ảnh khuôn mặt NGƯỜI A (rõ mặt, chính diện càng tốt):')
source_face_path_a = pick_one(files.upload(), 'ảnh mặt A')

print('\n>> Upload ảnh khuôn mặt NGƯỜI B:')
source_face_path_b = pick_one(files.upload(), 'ảnh mặt B')

print('\n>> Upload video mẫu (có 2 người, ví dụ cảnh hôn nhau):')
source_video_path = pick_one(files.upload(), 'video mẫu')

print(f'Ảnh mặt A: {source_face_path_a}')
print(f'Ảnh mặt B: {source_face_path_b}')
print(f'Video mẫu: {source_video_path}')

## 4. Khởi tạo model face analysis + face swapper

**Bước quan trọng:** cần xác định trong video, ai đứng bên trái / bên phải (theo frame đầu tiên có đủ 2 mặt) để biết gán ảnh A/B vào đúng người. Mặc định: **người A = mặt bên trái khung hình ở frame đầu tiên, người B = mặt bên phải**. Nếu bị ngược, đổi lại 2 ảnh upload ở bước 3 hoặc đảo `source_face_a`/`source_face_b` ở dưới.

### Fix riêng cho `onnxruntime-gpu` (có thể chưa có wheel cho Python bản mới trên Colab)

Cell dưới kiểm tra xem `onnxruntime` đã import được chưa. Nếu chưa, sẽ thử cài lại `onnxruntime-gpu` với log đầy đủ; nếu vẫn thất bại (do chưa có wheel tương thích Python 3.13), sẽ **fallback sang `onnxruntime` bản CPU** để pipeline vẫn chạy được (chỉ chậm hơn, không dùng được GPU cho bước detect/swap qua ONNX).

In [ ]:
import subprocess, sys, importlib

def try_import_onnxruntime():
    # Bắt Exception chứ không chỉ ImportError: onnxruntime-gpu thiếu libcudnn/libcublas
    # thường ném OSError/RuntimeError -> nếu chỉ bắt ImportError thì cell crash thay vì fallback.
    importlib.invalidate_caches()
    try:
        import onnxruntime
        print('onnxruntime OK, version:', onnxruntime.__version__)
        print('Available providers:', onnxruntime.get_available_providers())
        return True
    except Exception as e:
        print(f'Chưa import được onnxruntime: {type(e).__name__}: {e}')
        return False

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    return r.returncode

if not try_import_onnxruntime():
    # onnxruntime và onnxruntime-gpu dùng CHUNG namespace `cv2`-style: cùng package `onnxruntime`.
    # Cài cái này đè cái kia là trạng thái hỏng đã biết (mất CUDAExecutionProvider, import lỗi loạn)
    # -> luôn gỡ sạch cả hai trước khi cài lại.
    print('Gỡ sạch onnxruntime cũ rồi cài lại onnxruntime-gpu...')
    pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
    pip('install', 'onnxruntime-gpu==1.20.0')

    if not try_import_onnxruntime():
        print('onnxruntime-gpu không cài được (khả năng chưa có wheel cho bản Python này).')
        print('Fallback sang onnxruntime bản CPU...')
        pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
        pip('install', 'onnxruntime')
        assert try_import_onnxruntime(), 'Vẫn không cài được onnxruntime, xem log lỗi ở trên.'

In [ ]:
import cv2
import insightface
from insightface.app import FaceAnalysis

import onnxruntime
available_providers = onnxruntime.get_available_providers()
USE_CUDA = 'CUDAExecutionProvider' in available_providers
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if USE_CUDA else ['CPUExecutionProvider']
# ctx_id phải khớp với providers: 0 = GPU 0, -1 = CPU. Để nguyên 0 khi chỉ có CPU provider là mâu thuẫn.
ctx_id = 0 if USE_CUDA else -1
print('Dùng providers:', providers, '| ctx_id =', ctx_id)

app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=ctx_id, det_size=(640, 640))

swapper = insightface.model_zoo.get_model('/content/models/inswapper_128.onnx',
                                          download=False, providers=providers)

def load_source_face(path, label):
    # cv2.imread trả None khi file hỏng / định dạng không hỗ trợ (heic, webp lạ...).
    # Phải chặn ngay, không thì app.get(None) ném lỗi cv2 không nói gì về nguyên nhân thật.
    img = cv2.imread(path)
    assert img is not None, (
        f'Không đọc được ảnh {label} ({path}). Lưu lại thành .jpg/.png rồi upload lại.'
    )
    faces = app.get(img)
    assert len(faces) > 0, f'Không tìm thấy khuôn mặt trong ảnh {label}, thử ảnh khác rõ mặt hơn.'
    if len(faces) > 1:
        # Ảnh nguồn có nhiều mặt -> lấy mặt to nhất, vì thứ tự app.get() trả về không xác định.
        faces = sorted(faces,
                       key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]),
                       reverse=True)
        print(f'  (ảnh {label} có {len(faces)} mặt, dùng mặt lớn nhất)')
    return faces[0]

source_face_a = load_source_face(source_face_path_a, 'A')
source_face_b = load_source_face(source_face_path_b, 'B')

print('Đã detect khuôn mặt nguồn A và B thành công.')

### Fix riêng cho `basicsr` (bug với `torchvision` mới trên Colab)

Bản `torchvision` mới đã xoá hẳn module `torchvision.transforms.functional_tensor`, trong khi `basicsr` (dependency của GFPGAN) vẫn import `rgb_to_grayscale` theo đường cũ đó → `ModuleNotFoundError`. Hàm này nay nằm ở `torchvision.transforms.functional`.

Cell dưới định vị package bằng `importlib.util.find_spec` — hàm này chỉ **tìm** package chứ không chạy `__init__.py`, nên không dính đúng cái lỗi import mà ta đang muốn vá — rồi sửa mọi file `.py` còn tham chiếu module cũ (trong cả `basicsr`, `facexlib`, `gfpgan`).

Cell cũng tự **xoá các module hỏng khỏi `sys.modules`** cả trước lẫn sau khi vá — trước là để `find_spec` tra lại từ đĩa (một lần import hỏng trước đó có thể để lại `sys.modules['basicsr']` với `__spec__ = None`, làm `find_spec` ném lỗi), sau là để lần import kiểm chứng đọc đúng file vừa sửa. Nhờ vậy **không cần Runtime > Restart session** nữa: cell import GFPGAN bên dưới chạy được ngay.

In [ ]:
import importlib, importlib.util, sys, pathlib

OLD_MOD = 'torchvision.transforms.functional_tensor'
NEW_MOD = 'torchvision.transforms.functional'
PKGS = ('basicsr', 'facexlib', 'gfpgan')


def purge_modules():
    """Xoá các module (có thể đã import hỏng) khỏi cache.

    Phải chạy TRƯỚC find_spec: một lần import hỏng trước đó có thể để lại
    sys.modules['basicsr'] với __spec__ = None, khiến find_spec ném ValueError
    chứ không trả về None -> cell báo nhầm "chưa cài" dù basicsr đang có trên đĩa.
    Xoá xong thì find_spec tra lại từ đĩa, và cell import bên dưới cũng chạy
    được ngay mà không cần Runtime > Restart session.
    """
    for mod in [m for m in list(sys.modules) if m.split('.')[0] in PKGS]:
        del sys.modules[mod]
    importlib.invalidate_caches()


purge_modules()


def package_dir(name):
    """Trả về thư mục package mà KHÔNG import nó.

    find_spec() chỉ định vị package, không chạy __init__.py -> không dính đúng cái
    ModuleNotFoundError mà ta đang muốn vá. (Bản cũ dùng `find` trong %%bash: glob
    không khớp thư mục nào thì find trả 1, pipefail + set -e giết cell trước khi in gì.)
    """
    try:
        spec = importlib.util.find_spec(name)
    except Exception as e:
        print(f'  (không tra được {name}: {type(e).__name__}: {e})')
        return None
    if spec is None or not spec.submodule_search_locations:
        return None
    return pathlib.Path(list(spec.submodule_search_locations)[0])


targets = {}
for name in PKGS:
    d = package_dir(name)
    if d is None:
        print(f'{name:9s}: CHƯA CÀI')
    else:
        print(f'{name:9s}: {d}')
        targets[name] = d

assert 'basicsr' in targets, (
    'Không tìm thấy basicsr. Chạy lại cell cài basicsr ở mục 1 rồi chạy lại cell này.'
)

patched = []
for name, d in targets.items():
    for p in d.rglob('*.py'):
        try:
            text = p.read_text(encoding='utf-8')
        except (UnicodeDecodeError, OSError):
            continue
        if OLD_MOD not in text:
            continue
        p.write_text(text.replace(OLD_MOD, NEW_MOD), encoding='utf-8')
        patched.append(p)

print()
if patched:
    for p in patched:
        print(f'đã vá: {p}')
else:
    print('Không file nào cần vá (đã vá trước đó, hoặc torchvision bản này vẫn còn functional_tensor).')

# Purge lần nữa sau khi vá, để cell import bên dưới đọc lại file mới trên đĩa.
purge_modules()

# Kiểm chứng ngay tại đây thay vì để tới cell import gfpgan mới biết.
try:
    import basicsr.data.degradations
    print('\nOK: import basicsr.data.degradations thành công.')
except Exception as e:
    print(f'\nVẪN LỖI: {type(e).__name__}: {e}')
    print('Nếu lỗi vẫn liên quan tới torchvision, thử Runtime > Restart session rồi chạy lại cell này.')
    raise

In [ ]:
import subprocess, sys, importlib

def try_import_gfpgan():
    importlib.invalidate_caches()
    try:
        import gfpgan
        print('gfpgan OK, version:', getattr(gfpgan, '__version__', 'unknown'))
        return True
    except Exception as e:
        # Bắt Exception: gfpgan hỏng vì basicsr/torchvision thường ném ModuleNotFoundError,
        # nhưng tuỳ phiên bản torch cũng có thể là AttributeError/OSError.
        print(f'Chưa import được gfpgan: {type(e).__name__}: {e}')
        return False

GFPGAN_AVAILABLE = try_import_gfpgan()

if not GFPGAN_AVAILABLE:
    print('Thử cài lại gfpgan với log đầy đủ...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'gfpgan'],
                       capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    GFPGAN_AVAILABLE = try_import_gfpgan()

if not GFPGAN_AVAILABLE:
    print('gfpgan không cài được. Sẽ BỎ QUA bước phục hồi mặt (USE_GFPGAN tự tắt ở cell dưới).')
    print('Pipeline chính (swap mặt) vẫn chạy bình thường, chỉ là ảnh không được làm nét thêm.')

In [ ]:
# Tự tắt nếu gfpgan không cài được ở cell trên; đổi thành False để bỏ qua thủ công.
# globals().get(...): cho phép bỏ qua HẲN mục 5 (không chạy cell patch/import nào) mà vẫn
# chạy tiếp được pipeline chính, thay vì NameError: GFPGAN_AVAILABLE.
USE_GFPGAN = globals().get('GFPGAN_AVAILABLE', False)

restorer = None
if USE_GFPGAN:
    from gfpgan import GFPGANer
    restorer = GFPGANer(
        model_path='/content/models/GFPGANv1.4.pth',
        upscale=1,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None
    )
    print('GFPGAN sẵn sàng (chỉ chạy trên vùng crop quanh mặt đã swap).')
else:
    print('Bỏ qua GFPGAN, dùng ảnh swap gốc không phục hồi nét.')

## 6. Xử lý video: swap mặt từng frame (2 người, tracking theo danh tính)

**Vấn đề cần giải quyết:** model detect mặt mỗi frame **không đảm bảo thứ tự cố định** (frame này trả về [trái, phải], frame sau có thể trả về [phải, trái], đặc biệt khi 2 người quay đầu/che nhau lúc hôn). Nếu chỉ lấy theo index `[0]`, `[1]` thì mặt A/B sẽ bị **đảo lộn giữa các frame**, tạo hiệu ứng giật/lóe rất xấu.

**Cách xử lý ở đây — khớp theo *danh tính*, không chỉ theo vị trí:**

1. Ở frame đầu tiên detect được mặt, quy ước **A = mặt bên trái khung hình, B = mặt bên phải**, rồi lưu lại `normed_embedding` (vector nhận dạng ArcFace mà InsightFace **đã tính sẵn** ở bước detect — không tốn thêm chi phí) làm *danh tính tham chiếu* của từng người.
2. Mỗi frame sau, chi phí gán một khuôn mặt cho A/B = `(1 − cosine_similarity với embedding tham chiếu) + 0.25 × (khoảng cách vị trí đã chuẩn hoá)`. Embedding quyết định chính, vị trí chỉ phá hoà.
3. Mặt có `cosine_similarity < 0.20` bị **loại thẳng** — người lạ đi ngang hay người thứ 3 trong khung sẽ không bị swap nhầm (bản cũ luôn lấy mặt "gần nhất" nên không tránh được).
4. Việc gán được giải **tối ưu toàn cục** (duyệt hết các cách ghép, chỉ 2 người nên rất rẻ) thay vì greedy theo thứ tự A rồi B — greedy làm A luôn giành mặt trước, kể cả khi mặt đó rõ ràng là của B.

**Trường hợp 1 mặt bị che khuất (occlusion) hoàn toàn:** mặt hiện ra được gán cho **đúng người theo embedding**, người còn lại giữ nguyên frame gốc ở khung đó (không suy đoán mù).

**Người xuất hiện muộn:** nếu frame đầu chỉ detect được 1 mặt, người thứ hai vẫn được **khởi tạo muộn** ngay khi họ hiện ra ở frame sau.

**Ghi và ghép audio trong cùng một pass:** frame thô được ghi thẳng vào `ffmpeg` qua pipe, xuất ra H.264 kèm audio gốc. Bỏ được file trung gian `mp4v` (vốn làm mất chất lượng một lần trước khi re-encode lần hai) và bỏ luôn một lượt decode+encode toàn bộ video.

### 6.1 Bộ tracker (dùng chung cho cả chẩn đoán và xử lý)

Tách riêng khỏi vòng lặp xử lý để **cell chẩn đoán ở 6.2 chạy đúng logic đang dùng thật**, không phải bản chép lại dễ lệch. Chạy cell này trước, nó không đụng gì tới video nên rất nhanh.

**Ngưỡng tuyệt đối vs. so sánh tương đối.** Số đo thật từ video cho thấy lúc hôn, mặt người B nghiêng mạnh làm `sim` với chính danh tính của cô ấy tụt còn **+0.24**, chỉ cách ngưỡng loại (`MIN_SIM = 0.20`) đúng 0.04 — tụt thêm chút là cô ấy bị bỏ qua, mặt gốc lộ ra, nhấp nháy. Nhưng so sánh hai cách ghép thì lại cực chắc:

```
ghép đúng:  nam→A (+0.80) + nữ→B (+0.24) = 1.04
ghép đảo:   nam→B (−0.06) + nữ→A (+0.17) = 0.11   -> chênh 0.93
```

Giá trị tuyệt đối là thứ suy sụp khi mặt nghiêng; **thứ tự tương đối thì không**. Nên khi số mặt detect được đúng bằng số người đã biết — bài toán chỉ còn là "mặt nào là ai" — ngưỡng tuyệt đối được nới xuống `MIN_SIM_RELAXED` và để phép ghép tối ưu tự quyết. Ngưỡng chặt vẫn giữ nguyên cho trường hợp có mặt thừa, để còn lọc người lạ đi ngang.

In [ ]:
import itertools
import numpy as np

# ---------------- Tham số tracking ----------------
POS_W           = 0.25   # trọng số vị trí; embedding vẫn là yếu tố quyết định
MIN_SIM         = 0.20   # ngưỡng chặt: dùng khi có mặt thừa (lọc người lạ)
MIN_SIM_RELAXED = 0.10   # ngưỡng nới: dùng khi số mặt == số người đã biết.
                         # Không hạ thấp hơn: hai embedding của HAI NGƯỜI KHÁC NHAU
                         # dao động quanh 0 với độ lệch ~1/sqrt(512) = 0.044, nên 0.10
                         # (~2.3 sigma) vẫn chặn được người lạ, trong khi mặt nghiêng
                         # của chính chủ đo được thấp nhất là +0.24 — còn dư biên.
MAX_DIST_RATIO  = 0.15   # bán kính chuẩn hoá khoảng cách, theo cạnh lớn khung hình
EMB_EMA         = 0.05   # tốc độ cập nhật embedding tham chiếu
EMB_EMA_MIN_SIM = 0.50   # chỉ cập nhật khi khớp chắc -> tránh trôi sang nhầm người
UNASSIGNED_COST = 1.5    # phạt khi để một người không được gán

LABELS = ('A', 'B')


def face_center(face):
    x1, y1, x2, y2 = face.bbox
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0])


def face_width(face):
    return float(face.bbox[2] - face.bbox[0])


class FaceTracker:
    """Gán các khuôn mặt detect được ở mỗi frame cho đúng người A / B.

    Khớp theo danh tính (embedding ArcFace InsightFace đã tính sẵn), vị trí chỉ phá hoà.
    Việc gán được giải tối ưu toàn cục thay vì greedy theo thứ tự nhãn.
    """

    def __init__(self, frame_w, frame_h):
        self.max_dist = MAX_DIST_RATIO * max(frame_w, frame_h)
        self.ref_emb = {l: None for l in LABELS}
        self.last_pos = {l: None for l in LABELS}

    def sim_to(self, face, label):
        ref = self.ref_emb[label]
        if ref is None:
            return None
        return float(np.dot(face.normed_embedding, ref))

    def pos_cost(self, center, label):
        if self.last_pos[label] is None:
            return 1.0
        return min(float(np.linalg.norm(center - self.last_pos[label])) / self.max_dist, 1.0)

    def match_cost(self, face, center, label, min_sim=MIN_SIM):
        """Chi phí gán `face` cho `label`; None = không được phép gán."""
        s = self.sim_to(face, label)
        if s is None or s < min_sim:
            return None
        return (1.0 - s) + POS_W * self.pos_cost(center, label)

    def solve(self, labels, faces, centers, min_sim=MIN_SIM):
        """Gán tối ưu toàn cục (brute force — tối đa 2 người nên rất rẻ)."""
        best_total, best_map = None, {}
        idx_choices = [None] + list(range(len(faces)))
        for combo in itertools.product(idx_choices, repeat=len(labels)):
            used = [c for c in combo if c is not None]
            if len(set(used)) != len(used):
                continue                             # một mặt không thể là hai người
            total, mapping = 0.0, {}
            for label, idx in zip(labels, combo):
                if idx is None:
                    total += UNASSIGNED_COST
                    continue
                c = self.match_cost(faces[idx], centers[idx], label, min_sim)
                if c is None:
                    total = None
                    break
                total += c
                mapping[label] = idx
            if total is None:
                continue
            if best_total is None or total < best_total:
                best_total, best_map = total, mapping
        return best_map

    def step(self, faces):
        """Xử lý một frame. Trả về (assigned, debug).

        assigned thiếu nhãn = frame này không gán được người đó (giữ nguyên mặt gốc).
        debug chứa sim/cost của mọi mặt với mọi nhãn, đo TRƯỚC khi cập nhật state.
        """
        centers = [face_center(f) for f in faces]
        known = [l for l in LABELS if self.ref_emb[l] is not None]

        # Số mặt đúng bằng số người đã biết -> bài toán chỉ là "mặt nào là ai", một câu
        # hỏi TƯƠNG ĐỐI. Ngưỡng tuyệt đối lúc này là thứ mong manh nhất (mặt nghiêng làm
        # sim tụt sát ngưỡng dù cách ghép đúng vẫn hơn cách ghép đảo rất xa), nên nới ra.
        relaxed = bool(known) and len(faces) == len(known)
        min_sim = MIN_SIM_RELAXED if relaxed else MIN_SIM

        debug = {
            'centers': centers,
            'sim':  [{l: self.sim_to(f, l) for l in LABELS} for f in faces],
            'cost': [{l: self.match_cost(f, centers[i], l, min_sim) for l in LABELS}
                     for i, f in enumerate(faces)],
            'late_init': [],
            'min_sim': min_sim,
            'relaxed': relaxed,
        }

        assigned = self.solve(known, faces, centers, min_sim) if known else {}

        # Khởi tạo muộn: người xuất hiện sau frame đầu vẫn phải được nhận diện.
        unknown = [l for l in LABELS if self.ref_emb[l] is None]
        if unknown:
            taken = set(assigned.values())
            free = sorted((i for i in range(len(faces)) if i not in taken),
                          key=lambda i: centers[i][0])
            for label in unknown:                    # quy ước: A = bên trái, B = bên phải
                if not free:
                    break
                i = free.pop(0) if label == 'A' else free.pop(-1)
                assigned[label] = i
                self.ref_emb[label] = faces[i].normed_embedding.copy()
                debug['late_init'].append(label)

        for label, idx in assigned.items():
            self.last_pos[label] = centers[idx]
            # Cập nhật nhẹ embedding tham chiếu để bám theo góc mặt/ánh sáng,
            # nhưng chỉ khi khớp chắc chắn -> tránh trôi dần sang nhầm người.
            s = self.sim_to(faces[idx], label)
            if s is not None and s >= EMB_EMA_MIN_SIM:
                e = (1 - EMB_EMA) * self.ref_emb[label] + EMB_EMA * faces[idx].normed_embedding
                self.ref_emb[label] = e / (np.linalg.norm(e) + 1e-8)

        return assigned, debug


print(f'FaceTracker sẵn sàng.  POS_W={POS_W}  MIN_SIM={MIN_SIM} '
      f'(nới còn {MIN_SIM_RELAXED} khi số mặt khớp số người)')

### 6.1b Ghép mặt đã swap mà không để ai đè lên ai

Chỗ này quyết định pixel nào thuộc về ai. Bốn lớp:

**1. Không dùng mask mặc định của inswapper.** Nó dán về bằng cả ô vuông 128×128 đã align (`img_mask = img_white` trong source; mask tinh `fake_diff` được tính nhưng dòng dùng bị comment), rộng gấp ~1.5–1.7 lần bề ngang mặt → lúc hôn thì trùm sang mặt người kia. Ở đây tự ghép, giới hạn trong ellipse bám khuôn mặt.

**2. Mask phân vùng khuôn mặt** (`facexlib` parsenet, đã có sẵn). Loại nền, tóc, cổ, quần áo khỏi vùng dán.

**3. Nhường ở mức BỘ PHẬN, không phải mức pixel.** Đây là phần mới.

Nhường theo pixel làm ranh giới rơi vào chỗ hai mask cân bằng — một đường tuỳ tiện **có thể cắt ngang giữa cái mũi hoặc giữa cái môi**. Nửa cái mũi được swap, nửa kia không: mắt người đọc ra ngay là vỡ. Nhường theo bộ phận thì ranh giới chạy theo **đường viền giải phẫu**, thứ mắt người vốn quen nhìn.

Xác định bộ phận thuộc về ai bằng **vị trí giải phẫu**: crop đã được align theo template arcface, nên mũi của chính khuôn mặt đó **phải** nằm gần vị trí mũi chuẩn (x≈64, y≈72 trong crop 128). Mũi của người bên cạnh lọt vào crop sẽ nằm lệch hẳn. Với mỗi nhãn bộ phận, lấy các thành phần liên thông, giữ thành phần gần vị trí chuẩn nhất; các thành phần còn lại là **của người khác** và bị trừ khỏi vùng dán.

Nhờ vậy mũi A vẫn được swap (nó là của A), chỉ mask của B phải chừa chỗ đó ra. Nếu không thành phần nào đủ gần vị trí chuẩn thì bộ phận đó thành "của không ai" → bị loại khỏi cả hai → giữ pixel gốc.

**Giới hạn thật:** cách này chỉ tách được các bộ phận **rời nhau** (mũi, môi, mắt, chân mày). Vùng `skin` của hai người khi áp má thì **liền một khối**, thành phần liên thông không tách được — phần đó vẫn phải dựa vào ellipse và luật ở lớp 4.

**4. Vùng còn tranh chấp** — chủ yếu là da má chỗ tiếp xúc:

- chỉ một người đòi → người đó lấy
- cả hai đòi, một bên là bộ phận nhô ra của chính mình (mũi/môi) → bên đó nằm trước
- cả hai đòi ngang nhau → **không dán ai cả, giữ pixel gốc**

**5. Đang chạm nhau thì chừa hẳn mũi + miệng + môi ra** (`NO_SWAP_MOUTH_NOSE`).

Mỗi frame tính `khoảng cách 2 tâm mặt / bề ngang mặt`; dưới `CONTACT_RATIO` thì coi là đang chạm và trừ vùng mũi/môi của **cả hai** khỏi mask, giữ nguyên pixel gốc. Có điều kiện, nên frame nào hai người tách ra vẫn swap đầy đủ.

Vùng mũi/môi được dựng từ **template arcface** chứ không trông chờ parsenet. Lý do: số đo thật từ video cho thấy khi mặt nghiêng, parsenet trả về diện tích mũi/môi chỉ **0–115px** trong khi vùng thật khoảng **2000px**, và ở 5 frame chồng lấn nặng nhất thì ra **0 tuyệt đối** — trừ đi số 0 thì không chừa được gì. Crop đã align nên vị trí mũi/miệng là cố định, dựng hình học luôn chắc chắn hơn. parsenet vẫn được hợp nhất vào khi nó nhận ra, để thêm độ chính xác.

Các cờ tắt/bật: `USE_PART_OWNERSHIP` (lớp 3), `NO_SWAP_MOUTH_NOSE` (lớp 5), `MOUTH_NOSE_FROM_TEMPLATE` (nguồn dựng vùng mũi/môi).

In [ ]:
import numpy as np
import cv2

# ---------------- Cấu hình ghép ----------------
USE_PARSING         = True    # mask phân vùng khuôn mặt (facexlib parsenet)
USE_PART_OWNERSHIP  = True    # nhường ở mức bộ phận thay vì mức pixel
PROTRUDE_BONUS      = 1.0     # điểm cộng cho bộ phận nhô ra của chính mình khi tranh chấp
CONFLICT_BAND       = 0.35    # dưới mức chênh này coi như tranh chấp -> nhường pixel gốc

# Khi hai khuôn mặt đang chạm nhau thì CHỪA HẲN mũi + miệng + môi ra, giữ nguyên pixel gốc.
# Đây là vùng biến dạng nặng nhất lúc hôn. Đánh đổi: mũi và môi mang danh tính rất nặng,
# bỏ chúng đi thì kết quả bớt giống người nguồn — đặt False để so hai bản trên cùng video.
NO_SWAP_MOUTH_NOSE  = True
CONTACT_RATIO       = 1.00    # khoảng cách 2 tâm mặt / bề ngang mặt; nhỏ hơn = đang chạm.
                              # Số đo thật từ video: min 0.78, trung vị 0.88 -> 40/56 frame
                              # dưới ngưỡng, tức ngưỡng này kích hoạt đúng.

# Vùng mũi/môi dựng từ TEMPLATE thay vì trông chờ parsenet nhận ra.
# Số đo thật: khi mặt nghiêng, parsenet cho diện tích mũi/môi chỉ 0-115px trong khi
# vùng thật khoảng 2000px — ở 5 frame chồng lấn nặng nhất thì ra 0 tuyệt đối. Trừ đi
# số 0 thì không chừa được gì, nên không thể chỉ dựa vào nó.
MOUTH_NOSE_FROM_TEMPLATE = True
MOUTH_NOSE_W        = 0.55    # bán trục ngang, tính theo khoảng cách 2 mắt trong crop
MOUTH_NOSE_H        = 0.60    # bán trục dọc

# Mặt nghiêng sâu quá thì MỜ DẦN rồi thôi không dán.
# inswapper_128 huấn luyện chủ yếu trên mặt gần chính diện, nghiêng mạnh thì nó
# "làm phẳng" khuôn mặt về hướng chính diện -> dán lên đầu đang nghiêng nhìn ra ngay
# là giả. Qua một mức nào đó, giữ mặt gốc trông tự nhiên hơn là dán.
# Dùng dải MỜ DẦN chứ không cắt phựt, để không bị giật khi frame vượt ngưỡng.
# MẶC ĐỊNH TẮT. Số đo thật từ video cho thấy với ngưỡng ban đầu, lớp này tắt swap trên
# 32/56 frame của A và 44/56 của B — phá video chứ không cứu. Bật lên sau khi đã xem
# phân bố yaw thật ở cell chẩn đoán và chọn ngưỡng theo số đo.
YAW_FADE_ON         = False
YAW_FULL            = 0.35    # dưới mức này: dán đầy đủ  (thang [0,1])
YAW_NONE            = 0.75    # trên mức này: không dán nữa

# Tỉ lệ (khoảng cách 2 mắt) / (mắt->miệng) của mặt chính diện chuẩn, tính từ template
# arcface: 35.2377 / 40.6866 = 0.86608. Dùng làm mốc để quy độ nghiêng về thang [0,1].
FRONTAL_IOD_RATIO   = 0.86608
FOREIGN_DILATE      = 0.02    # nới vùng bộ phận của người khác, theo cạnh crop
PART_MAX_DIST       = 0.25    # bộ phận lệch quá bao nhiêu (theo cạnh crop) thì coi là của người khác
PARSE_SIZE          = 512
SWAP_SIZE           = 128

# Nhãn parsenet (CelebAMask-HQ 19 lớp). MASK_COLORMAP lấy nguyên từ facexlib:
# giữ da + mắt + mũi + môi + tai, bỏ nền(0) / cổ(14) / quần áo(16) / tóc(17) / mũ(18).
_MASK_COLORMAP = [0, 255, 255, 255, 255, 255, 255, 255, 255, 255,
                  255, 255, 255, 255, 0, 255, 0, 0, 0]
FACE_LABELS     = [i for i, c in enumerate(_MASK_COLORMAP) if c == 255]
PROTRUDE_LABELS = [10, 11, 12, 13]        # nose, mouth, upper lip, lower lip
HAIR_LABELS     = [17, 18]                # hair, hat

# inswapper SINH RA cả ô 128x128, trong đó có tóc của người nguồn ở rìa. Nếu vùng tóc
# lọt vào mask thì tóc lạ bị dán vào video. Không thể chỉ trông vào FACE_LABELS: ở mặt
# nghiêng parsenet nhận tóc chập chờn, và ellipse dựng theo tỉ lệ template thì vươn ra
# quá mép mặt thật (mặt nghiêng hẹp hơn template giả định).
HAIR_DILATE     = 0.03    # nới vùng tóc trước khi trừ, theo cạnh crop
MASK_ERODE      = 0.02    # co viền mask vào trong — chặn rò rỉ kể cả khi parsenet
                          # không nhận ra tóc. Đặt 0 nếu thấy mất nhiều phần mặt.

# Bộ phận nào neo vào mốc giải phẫu nào. Chỉ liệt kê các bộ phận RỜI NHAU —
# `skin` của hai người khi áp má là một khối liền, không tách được kiểu này.
PART_ANCHOR = {10: 'nose', 11: 'mouth', 12: 'mouth', 13: 'mouth',
               4: 'l_eye', 5: 'r_eye', 2: 'l_eye', 3: 'r_eye'}

_PARSER = None
_PARSE_DEVICE = None
_WARNED = set()


def _warn_once(key, msg):
    """In cảnh báo một lần thôi — vòng lặp chạy hàng trăm frame."""
    if key not in _WARNED:
        _WARNED.add(key)
        print(msg)


def canon_points(size=SWAP_SIZE):
    """Vị trí giải phẫu chuẩn trong crop đã align, suy thẳng từ template arcface.

    estimate_norm với image_size chia hết cho 128 dùng ratio = size/128 và dịch x thêm
    8*ratio. Với crop 128: hai mắt (46.3, 51.7) và (81.5, 51.5), mũi (64.0, 71.7),
    hai khoé miệng (49.6, 92.4) và (78.7, 92.2).
    """
    s = size / 128.0
    return {
        'l_eye': (46.29 * s, 51.70 * s),
        'r_eye': (81.53 * s, 51.50 * s),
        'nose':  (64.03 * s, 71.74 * s),
        'mouth': (64.14 * s, 92.28 * s),
    }


def split_own_foreign(labels, size):
    """Tách bộ phận CỦA khuôn mặt ở giữa crop khỏi bộ phận của người bên cạnh.

    Crop đã align nên mỗi bộ phận phải nằm gần vị trí chuẩn của nó. Với mỗi nhãn,
    lấy các thành phần liên thông: thành phần gần mốc chuẩn nhất (và đủ gần) là của
    khuôn mặt này; các thành phần khác là của người bên cạnh lọt vào crop.

    Trả về (own_protrude, foreign) — own_protrude chỉ gồm mũi/môi của chính mình,
    dùng để phá hoà ở bước sau; foreign là vùng phải trừ khỏi mask.
    """
    canon = canon_points(size)
    max_d = PART_MAX_DIST * size
    min_area = max(int((size / 32.0) ** 2), 8)      # bỏ đốm nhiễu li ti
    own_prot = np.zeros((size, size), np.float32)
    foreign = np.zeros((size, size), np.float32)

    for lab, anchor in PART_ANCHOR.items():
        blob = (labels == lab).astype(np.uint8)
        if not blob.any():
            continue
        n, cc, stats, cents = cv2.connectedComponentsWithStats(blob, 8)
        ax, ay = canon[anchor]

        best, best_d = None, None
        for k in range(1, n):
            if stats[k, cv2.CC_STAT_AREA] < min_area:
                continue
            d = float(np.hypot(cents[k][0] - ax, cents[k][1] - ay))
            if best_d is None or d < best_d:
                best, best_d = k, d

        for k in range(1, n):
            if stats[k, cv2.CC_STAT_AREA] < min_area:
                continue
            comp = (cc == k)
            if k == best and best_d is not None and best_d <= max_d:
                if lab in PROTRUDE_LABELS:
                    own_prot[comp] = 1.0
            else:
                # Không đủ gần mốc chuẩn -> của người bên cạnh (hoặc không của ai).
                foreign[comp] = 1.0

    if foreign.any():
        k = max(int(FOREIGN_DILATE * size), 1)
        foreign = cv2.dilate(foreign, np.ones((k, k), np.uint8), iterations=1)
    return own_prot, foreign


def yaw_score(face):
    """Độ quay ngang của đầu, suy từ 5 điểm mốc. 0 = chính diện, 1 = nghiêng hết cỡ.

    Quay ngang làm khoảng cách hai mắt CO LẠI (phối cảnh), trong khi khoảng cách
    mắt->miệng gần như không đổi. Lấy tỉ số hai đại lượng đó rồi quy về thang [0,1]
    theo mốc chính diện của template arcface.

    Cách cũ (chiếu mũi lên đường nối hai mắt, chia cho khoảng cách hai mắt) đã bị bỏ:
    mẫu số co về 0 khi nghiêng sâu nên giá trị nổ tung — đo trên video thật ra tới
    12.78, khiến ngưỡng không thể đặt được. Cách này bị chặn nên đặt ngưỡng mới có nghĩa.

    Vẫn BẤT BIẾN với nghiêng đầu (roll), kích thước và vị trí — cả hai đại lượng đều
    là độ dài. insightface 0.7.3 không có face.pose (Face.__getattr__ trả None âm thầm)
    nên phải tự tính.
    """
    kps = np.asarray(face.kps, dtype=np.float64)
    vertical = float(np.linalg.norm((kps[0] + kps[1]) / 2.0 - (kps[3] + kps[4]) / 2.0))
    if vertical < 1e-6:
        return 0.0
    iod = float(np.linalg.norm(kps[1] - kps[0]))
    return float(np.clip(1.0 - (iod / vertical) / FRONTAL_IOD_RATIO, 0.0, 1.0))


def yaw_weight(score):
    """Độ đậm của lớp dán theo độ nghiêng: 1 = dán đầy đủ, 0 = không dán."""
    if not YAW_FADE_ON or YAW_NONE <= YAW_FULL:
        return 1.0
    return float(np.clip((YAW_NONE - score) / (YAW_NONE - YAW_FULL), 0.0, 1.0))


def mouth_nose_region(size=SWAP_SIZE):
    """Vùng mũi + miệng + môi, suy thẳng từ template arcface. Luôn có, không phụ
    thuộc parsenet có nhận ra hay không.

    Crop đã align nên các mốc này cố định: mũi (64.0, 71.7), miệng (64.1, 92.3),
    hai mắt cách nhau 35.2px (với crop 128). Ellipse bao từ sống mũi xuống dưới môi.
    """
    cp = canon_points(size)
    iod = cp['r_eye'][0] - cp['l_eye'][0]
    cx = (cp['nose'][0] + cp['mouth'][0]) / 2.0
    cy = (cp['nose'][1] + cp['mouth'][1]) / 2.0
    m = np.zeros((size, size), np.float32)
    cv2.ellipse(m, (int(round(cx)), int(round(cy))),
                (max(int(MOUTH_NOSE_W * iod), 1), max(int(MOUTH_NOSE_H * iod), 1)),
                0, 0, 360, 1.0, -1)
    return m


def build_face_mask(size, scale=1.0):
    """Ellipse bám khuôn mặt, trong không gian crop đã align."""
    m = np.zeros((size, size), np.float32)
    ax = min(int(size * 0.33 * scale), size // 2 - 1)
    ay = min(int(size * 0.47 * scale), size // 2 - 1)
    cv2.ellipse(m, (int(size * 0.50), int(size * 0.53)), (ax, ay), 0, 0, 360, 1.0, -1)
    return m


def init_parser():
    global _PARSER, _PARSE_DEVICE
    if _PARSER is not None:
        return _PARSER
    import torch
    from facexlib.parsing import init_parsing_model
    _PARSE_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    _PARSER = init_parsing_model(model_name='parsenet', device=_PARSE_DEVICE)
    _PARSER.eval()
    print(f'  parsenet đã nạp trên {_PARSE_DEVICE}')
    return _PARSER


def parse_crop(aimg):
    """Phân vùng crop đã align, trả về BẢN ĐỒ NHÃN (19 lớp) cùng cỡ với aimg.

    Tiền xử lý chép theo facexlib FaceRestoreHelper để khớp lúc model được train.
    """
    import torch
    parser = init_parser()
    x = cv2.resize(aimg, (PARSE_SIZE, PARSE_SIZE), interpolation=cv2.INTER_LINEAR)
    x = x.astype(np.float32)[:, :, ::-1] / 255.0            # BGR -> RGB, [0,1]
    x = torch.from_numpy(np.ascontiguousarray(x)).permute(2, 0, 1)
    x = ((x - 0.5) / 0.5).unsqueeze(0).to(_PARSE_DEVICE)     # normalize(0.5, 0.5)
    with torch.no_grad():
        labels = parser(x)[0].argmax(dim=1).squeeze().cpu().numpy().astype(np.uint8)

    t = max(PARSE_SIZE // 50, 4)                             # bỏ viền đen do warp
    labels[:t, :] = labels[-t:, :] = 0
    labels[:, :t] = labels[:, -t:] = 0
    size = aimg.shape[0]
    return cv2.resize(labels, (size, size), interpolation=cv2.INTER_NEAREST)


def build_masks_for(frame, face, M, size=SWAP_SIZE):
    """Mask vùng dán cho một khuôn mặt. Trả về (mask, mask_nhô_ra_của_mình, tên_nguồn).

    Luôn cắt crop từ `frame` GỐC để mask phản ánh đúng cảnh thật ai đang che ai.
    """
    ell = build_face_mask(size)
    # Vùng mũi/môi hình học luôn dựng được; parsenet chỉ bổ sung độ chính xác khi nó
    # thật sự nhận ra. Hợp nhất hai nguồn để tính năng chừa mũi/môi không phụ thuộc
    # vào việc parsenet có hoạt động ở góc mặt đó hay không.
    base_prot = mouth_nose_region(size) if MOUTH_NOSE_FROM_TEMPLATE         else np.zeros((size, size), np.float32)

    if not USE_PARSING:
        return ell, base_prot, 'ellipse'

    try:
        aimg = cv2.warpAffine(frame, M, (size, size), borderValue=0.0)
        labels = parse_crop(aimg)
    except Exception as e:
        _warn_once('parse', f'  parsenet không dùng được ({type(e).__name__}: {e}) '
                            f'-> dùng ellipse')
        return ell, base_prot, 'ellipse'

    mask = ell * np.isin(labels, FACE_LABELS).astype(np.float32)

    # Trừ tóc CÓ NỚI RỘNG: chỉ loại theo nhãn là chưa đủ, vì viền tóc-da hay bị gán
    # nhầm thành da, mà đúng chỗ đó là nơi tóc người nguồn rò vào.
    hair = np.isin(labels, HAIR_LABELS).astype(np.float32)
    if hair.any() and HAIR_DILATE > 0:
        k = max(int(HAIR_DILATE * size), 1)
        hair = cv2.dilate(hair, np.ones((k, k), np.uint8), iterations=1)
    mask = mask * (1.0 - hair)

    # Co viền mask: hoạt động kể cả khi parsenet KHÔNG nhận ra tóc — trường hợp
    # đã thấy ở mặt nghiêng. Đổi lại thì mất một dải mỏng ở rìa khuôn mặt.
    if MASK_ERODE > 0:
        k = max(int(MASK_ERODE * size), 1)
        mask = cv2.erode(mask, np.ones((k, k), np.uint8), iterations=1)

    if not USE_PART_OWNERSHIP:
        prot = np.maximum(base_prot,
                          np.isin(labels, PROTRUDE_LABELS).astype(np.float32))
        return mask, prot, 'parsenet'

    own_prot, foreign = split_own_foreign(labels, size)
    mask = mask * (1.0 - np.clip(foreign, 0.0, 1.0))    # chừa bộ phận của người bên cạnh
    return mask, np.maximum(base_prot, own_prot), 'parsenet+parts'


def combine_masks(face_masks, prot_masks):
    """Quyết định pixel nào thuộc về ai ở vùng còn chồng nhau (chủ yếu là da má).

    Chỗ hai bên đòi ngang nhau thì trọng số của CẢ HAI về 0 -> giữ pixel gốc,
    thay vì để một bên xoá mất bộ phận của bên kia.
    """
    if len(face_masks) == 1:
        return face_masks
    F = np.stack(face_masks)
    score = F + PROTRUDE_BONUS * np.stack(prot_masks)
    out = []
    for i in range(len(face_masks)):
        other = np.max(np.delete(score, i, axis=0), axis=0)
        dom = score[i] - other                        # >0 = mình đòi mạnh hơn
        out.append(F[i] * np.clip(dom / CONFLICT_BAND, 0.0, 1.0))
    return out


def swap_all_faces(frame, assigned, target_faces, swapper, source_face):
    """Swap mọi người trong một frame rồi ghép lại, không ai đè lên ai.

    frame : khung GỐC (chưa swap gì). Gọi swapper tuần tự trên khung đã swap khiến
            lần thứ hai cắt crop từ vùng lần thứ nhất vừa ghi đè lên chính mặt nó.
    """
    if not assigned:
        return frame

    h, w = frame.shape[:2]
    fakes, faces_m, prots_m, centers, widths, yaws = [], [], [], [], [], []
    for label, idx in assigned.items():
        face = target_faces[idx]
        yaws.append(yaw_score(face))
        bgr_fake, M = swapper.get(frame, face, source_face[label], paste_back=False)
        IM = cv2.invertAffineTransform(M)
        mask, prot, _ = build_masks_for(frame, face, M, bgr_fake.shape[0])

        fakes.append(cv2.warpAffine(bgr_fake, IM, (w, h), borderValue=0.0).astype(np.float32))
        faces_m.append(cv2.warpAffine(mask, IM, (w, h), borderValue=0.0))
        prots_m.append(cv2.warpAffine(prot, IM, (w, h), borderValue=0.0))
        centers.append(face_center(face))
        widths.append(face_width(face))

    # Làm mềm viền TRƯỚC khi so sánh, để ranh giới chuyển tiếp mượt thay vì răng cưa
    feather = max(2.0, 0.04 * float(np.mean(widths)))
    faces_m = [cv2.GaussianBlur(m, (0, 0), sigmaX=feather) for m in faces_m]
    prots_m = [cv2.GaussianBlur(m, (0, 0), sigmaX=feather) for m in prots_m]

    # Đang chạm nhau -> chừa mũi + môi ra. Chỉ áp cho frame có tiếp xúc, để những frame
    # còn lại vẫn swap đầy đủ và giữ được danh tính.
    if NO_SWAP_MOUTH_NOSE and len(faces_m) > 1:
        mean_w = float(np.mean(widths))
        for i in range(len(faces_m)):
            touching = any(
                float(np.linalg.norm(centers[i] - centers[j])) / max(mean_w, 1e-6) < CONTACT_RATIO
                for j in range(len(faces_m)) if j != i
            )
            if touching:
                faces_m[i] = faces_m[i] * (1.0 - np.clip(prots_m[i], 0.0, 1.0))

    # Nghiêng sâu quá -> mờ dần rồi thôi không dán, để không có cảm giác dán mặt phẳng
    # lên cái đầu đang nghiêng.
    if YAW_FADE_ON:
        for i in range(len(faces_m)):
            wgt = yaw_weight(yaws[i])
            if wgt < 1.0:
                faces_m[i] = faces_m[i] * wgt
                prots_m[i] = prots_m[i] * wgt

    out = frame.astype(np.float32)
    for m, fk in zip(combine_masks(faces_m, prots_m), fakes):
        if float(m.max()) <= 0.0:
            continue
        out = m[..., None] * fk + (1.0 - m[..., None]) * out
    return np.clip(out, 0, 255).astype(np.uint8)


print(f'swap_all_faces sẵn sàng.  USE_PARSING={USE_PARSING}  '
      f'USE_PART_OWNERSHIP={USE_PART_OWNERSHIP}  '
      f'NO_SWAP_MOUTH_NOSE={NO_SWAP_MOUTH_NOSE} (ngưỡng chạm {CONTACT_RATIO}, '
      f'vùng mũi/môi từ template={MOUTH_NOSE_FROM_TEMPLATE})')
if YAW_FADE_ON:
    print(f'  Mặt nghiêng: dán đầy đủ tới yaw={YAW_FULL}, mờ dần, thôi hẳn từ yaw={YAW_NONE}')
else:
    print('  Mặt nghiêng: lớp mờ dần đang TẮT (xem phân bố yaw ở cell 6.2 rồi hãy bật)')

### 6.2 Chẩn đoán: đo số liệu thật từ video

Cell này **chỉ đọc, không tạo video**. Nó trả lời bốn câu hỏi, câu 3 và 4 là mới:

1. **Tracking có gán nhầm không?** `margin` = sim với ref của mình − sim với ref người kia. Về 0 hoặc âm nghĩa là chọn A/B đã thành tung đồng xu.
2. **Vùng paste-back mặc định tràn bao nhiêu?** Con số *nếu không sửa gì*, để đối chiếu.
3. **Ngưỡng "đang chạm" có kích hoạt không?** In khoảng cách 2 tâm mặt / bề ngang mặt của từng frame, so với `CONTACT_RATIO`, và đếm bao nhiêu frame vượt/không vượt. Nếu không frame nào dưới ngưỡng thì `NO_SWAP_MOUTH_NOSE` **chưa từng chạy** — và đó là lý do mũi/môi vẫn bị swap.
4. **Vùng mũi/môi có được nhận ra không?** Gọi thẳng `build_masks_for()` của cell 6.1b và in **diện tích `prot`** (vùng mũi + miệng + môi của chính khuôn mặt đó). `prot = 0` nghĩa là parsenet không nhận ra mũi/môi ở góc mặt này — lúc đó dù ngưỡng có kích hoạt cũng không có gì để trừ.

Hai câu cuối phân biệt được ba nguyên nhân khác nhau của cùng một triệu chứng "mũi môi vẫn bị swap": ngưỡng không kích hoạt / parsenet không thấy mũi môi / cả hai đều ổn nhưng vấn đề nằm chỗ khác.

Ảnh lưu ra tô **mask thật** (vùng sẽ được dán) theo màu nhãn, và khoanh **viền trắng quanh vùng mũi/môi** sẽ bị chừa ra khi chạm nhau. Không thấy viền trắng = parsenet không nhận ra vùng đó.

Chỉnh `DIAG_END` về vài trăm frame quanh cảnh hôn để chạy nhanh. Tracker luôn chạy từ frame 0 để trạng thái giống hệt lần xử lý thật.

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from insightface.utils import face_align

# ---------------- Cấu hình chẩn đoán ----------------
DIAG_START     = 0
DIAG_END       = None     # None = hết video
DIAG_SAVE_N    = 10       # lưu bao nhiêu frame đáng ngờ nhất ra ảnh
DIAG_DIR       = '/content/diag'
DIAG_DRAW_MASK = True     # tô mask thật (gọi build_masks_for của 6.1b)

os.makedirs(DIAG_DIR, exist_ok=True)
LABEL_COLORS = {'A': (255, 140, 0), 'B': (0, 140, 255)}      # BGR

assert 'build_masks_for' in globals() and 'SWAP_SIZE' in globals(), (
    'Chạy cell 6.1b (Ghép mặt) trước cell này.'
)


def paste_quad(face, size=SWAP_SIZE):
    """Ô vuông inswapper sẽ dán về NẾU dùng paste_back mặc định — để ĐO vấn đề
    của cách mặc định, KHÔNG phải mask pipeline đang dùng."""
    M = face_align.estimate_norm(face.kps, size)
    IM = cv2.invertAffineTransform(M)
    corners = np.array([[0, 0], [size, 0], [size, size], [0, size]], np.float32)
    quad = cv2.transform(corners.reshape(1, -1, 2), IM).reshape(-1, 2)
    w, h = float(np.ptp(quad[:, 0])), float(np.ptp(quad[:, 1]))
    ms = float(np.sqrt(max(w * h, 1.0)))
    k = max(int(ms) // 10, 10)
    ctr = quad.mean(0)
    return (ctr + (quad - ctr) * (1.0 - (k / 2.0) / (ms / 2.0))).astype(np.float32)


def bbox_poly(face):
    x1, y1, x2, y2 = [float(v) for v in face.bbox]
    return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], np.float32)


def covered_frac(poly, target):
    inter, _ = cv2.intersectConvexConvex(poly, target)
    area = cv2.contourArea(target)
    return float(inter / area) if area > 0 else 0.0


def fmt(v, nd=2):
    return '  --' if v is None else f'{v:+.{nd}f}'


def real_masks(frame, faces, assigned):
    """Mask THẬT pipeline dùng + vùng mũi/môi (prot), đưa về toạ độ khung hình."""
    h, w = frame.shape[:2]
    masks, prots, srcs, hairs = {}, {}, {}, {}
    for label, idx in assigned.items():
        f = faces[idx]
        M = face_align.estimate_norm(f.kps, SWAP_SIZE)
        IM = cv2.invertAffineTransform(M)
        m, p, src = build_masks_for(frame, f, M, SWAP_SIZE)
        masks[label] = cv2.warpAffine(m, IM, (w, h), borderValue=0.0)
        prots[label] = cv2.warpAffine(p, IM, (w, h), borderValue=0.0)
        srcs[label] = src
        # Diện tích parsenet gán là TÓC trong crop. Gần 0 ở mặt nghiêng nghĩa là nó
        # không thấy tóc -> tóc người nguồn có đường rò vào mask.
        try:
            aimg = cv2.warpAffine(frame, M, (SWAP_SIZE, SWAP_SIZE), borderValue=0.0)
            hairs[label] = float(np.isin(parse_crop(aimg), HAIR_LABELS).sum())
        except Exception:
            hairs[label] = None
    return masks, prots, srcs, hairs


def annotate(frame, faces, assigned, dbg, masks=None, prots=None):
    """mask thật = vùng tô màu | mũi/môi = viền trắng | bbox = khung đậm."""
    vis = frame.astype(np.float32)
    if masks:
        for label, m in masks.items():
            a = np.clip(m, 0.0, 1.0)[..., None] * 0.45
            vis = vis * (1 - a) + a * np.array(LABEL_COLORS[label], np.float32)
    vis = np.clip(vis, 0, 255).astype(np.uint8)

    if prots:
        for label, p in prots.items():
            binm = (np.clip(p, 0.0, 1.0) > 0.4).astype(np.uint8)
            if binm.any():
                cnts, _ = cv2.findContours(binm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(vis, cnts, -1, (255, 255, 255), 2)

    owner = {idx: label for label, idx in assigned.items()}
    for i, f in enumerate(faces):
        label = owner.get(i)
        col = LABEL_COLORS.get(label, (150, 150, 150))
        x1, y1, x2, y2 = f.bbox.astype(int)
        cv2.rectangle(vis, (x1, y1), (x2, y2), col, 2)
        cv2.polylines(vis, [paste_quad(f).astype(np.int32)], True, col, 1)
        s = dbg['sim'][i]
        cv2.putText(vis,
                    f"{label or '?'} simA={fmt(s['A'])} simB={fmt(s['B'])} det={f.det_score:.2f}",
                    (x1, max(16, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1, cv2.LINE_AA)
    return vis


# ---------------- Quét video ----------------
cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'
ret, probe = cap.read()
assert ret, 'Không đọc được frame nào.'
fh, fw = probe.shape[:2]
cap.release()

cap = cv2.VideoCapture(source_video_path)
tracker = FaceTracker(fw, fh)
rows, keep = [], []
idx = -1
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
pbar = tqdm(total=(DIAG_END if DIAG_END is not None else total), unit='frame', desc='chẩn đoán')
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        idx += 1
        if DIAG_END is not None and idx > DIAG_END:
            break

        faces = app.get(frame)
        assigned, dbg = tracker.step(faces)     # luôn chạy từ frame 0
        pbar.update(1)
        if idx < DIAG_START:
            continue

        iA, iB = assigned.get('A'), assigned.get('B')
        row = {'frame': idx, 'n_faces': len(faces),
               'co_A': iA is not None, 'co_B': iB is not None}
        for label, i, other in (('A', iA, 'B'), ('B', iB, 'A')):
            row[f'yaw_{label}'] = yaw_score(faces[i]) if i is not None else None
            row[f'sim_{label}'] = dbg['sim'][i][label] if i is not None else None
            row[f'det_{label}'] = float(faces[i].det_score) if i is not None else None
            if (i is not None and dbg['sim'][i][label] is not None
                    and dbg['sim'][i][other] is not None):
                row[f'margin_{label}'] = dbg['sim'][i][label] - dbg['sim'][i][other]
            else:
                row[f'margin_{label}'] = None

        suspect = 0.0
        if iA is not None and iB is not None:
            fA, fB = faces[iA], faces[iB]
            ovBA = covered_frac(paste_quad(fB), bbox_poly(fA))
            ovAB = covered_frac(paste_quad(fA), bbox_poly(fB))
            d = float(np.linalg.norm(face_center(fA) - face_center(fB)))
            mw = (face_width(fA) + face_width(fB)) / 2.0
            ratio = d / mw if mw > 0 else None
            row['B_phu_len_mat_A'] = ovBA
            row['A_phu_len_mat_B'] = ovAB
            row['kc_2_mat_theo_be_ngang'] = ratio
            # Đúng điều kiện mà swap_all_faces dùng để quyết định chừa mũi/môi
            row['dang_cham'] = (ratio is not None and ratio < CONTACT_RATIO)
            suspect = max(ovBA, ovAB)
        else:
            row['B_phu_len_mat_A'] = row['A_phu_len_mat_B'] = None
            row['kc_2_mat_theo_be_ngang'] = None
            row['dang_cham'] = None
            if faces:
                suspect = 0.9

        for label in ('A', 'B'):
            m = row.get(f'margin_{label}')
            if m is not None and m < 0.25:
                suspect = max(suspect, 1.2 - m)

        rows.append(row)
        if suspect > 0.15:
            keep.append((suspect, idx, frame.copy(), faces, assigned, dbg))
            keep.sort(key=lambda t: -t[0])
            del keep[DIAG_SAVE_N:]
finally:
    pbar.close()
    cap.release()

print(f'\nĐã ghi số liệu {len(rows)} frame (video {fw}x{fh}).')

# ---------------- Ngưỡng chạm có kích hoạt không? ----------------
ratios = [r['kc_2_mat_theo_be_ngang'] for r in rows
          if r.get('kc_2_mat_theo_be_ngang') is not None]
print('\n' + '=' * 70)
print(f'NGƯỠNG CHẠM  (CONTACT_RATIO = {CONTACT_RATIO}, NO_SWAP_MOUTH_NOSE = {NO_SWAP_MOUTH_NOSE})')
print('=' * 70)
if not ratios:
    print('  Không frame nào detect đủ 2 người -> ngưỡng không bao giờ xét tới.')
else:
    fired = sum(1 for r in ratios if r < CONTACT_RATIO)
    a = np.array(ratios)
    print(f'  kc 2 tâm mặt / bề ngang mặt:  nhỏ nhất {a.min():.2f}  '
          f'trung vị {np.median(a):.2f}  lớn nhất {a.max():.2f}')
    print(f'  Số frame dưới ngưỡng (coi là đang chạm): {fired}/{len(ratios)} ({fired/len(ratios):.0%})')
    if fired == 0:
        print(f'  -> NGƯỠNG CHƯA TỪNG KÍCH HOẠT. Mũi/môi không bao giờ được chừa ra.')
        print(f'     Nâng CONTACT_RATIO lên khoảng {a.min() + 0.05:.2f} thì mới bắt được cảnh hôn.')
    else:
        print(f'  -> Ngưỡng CÓ kích hoạt ở {fired} frame.')

# ---------------- Mặt nghiêng bao nhiêu? ----------------
print()
print('=' * 70)
print(f'ĐỘ NGHIÊNG MẶT  (YAW_FADE_ON = {YAW_FADE_ON}, dán đủ tới {YAW_FULL}, '
      f'thôi hẳn từ {YAW_NONE})')
print('=' * 70)
print('  yaw = 0 là chính diện; càng lớn càng nghiêng. Bất biến với nghiêng đầu và cỡ mặt.')
for label in ('A', 'B'):
    ys = [r[f'yaw_{label}'] for r in rows if r.get(f'yaw_{label}') is not None]
    if not ys:
        print(f'  {label}: không có dữ liệu')
        continue
    a = np.array(ys)
    full = int((a <= YAW_FULL).sum())
    none = int((a >= YAW_NONE).sum())
    fade = len(a) - full - none
    print(f'  {label}: nhỏ nhất {a.min():.2f}  trung vị {np.median(a):.2f}  '
          f'lớn nhất {a.max():.2f}')
    print(f'      dán đầy đủ {full} frame | mờ dần {fade} frame | không dán {none} frame')
mx = [r.get('yaw_A') for r in rows if r.get('yaw_A') is not None]
if mx:
    hi = max(mx + [r['yaw_B'] for r in rows if r.get('yaw_B') is not None])
    if hi <= YAW_FULL:
        print(f'  -> Cả video đều dưới {YAW_FULL}: lớp mờ theo độ nghiêng CHƯA từng chạy.')
        print(f'     Muốn nó tác dụng thì hạ YAW_FULL xuống dưới {hi:.2f}.')

# ---------------- Mask thật + vùng mũi/môi ----------------
mask_rows = []
for _, i, frm, fcs, asg, dbgi in sorted(keep, key=lambda t: t[1]):
    masks = prots = None
    if DIAG_DRAW_MASK and asg:
        masks, prots, srcs, hairs = real_masks(frm, fcs, asg)
        rec = {'frame': i, 'nguon': '/'.join(sorted(set(srcs.values())))}
        for lb in ('A', 'B'):
            rec[f'mask_{lb}'] = float(masks[lb].sum()) if lb in masks else 0.0
            rec[f'prot_{lb}'] = float(prots[lb].sum()) if lb in prots else 0.0
            rec[f'hair_{lb}'] = hairs.get(lb)
        r0 = next((r for r in rows if r['frame'] == i), {})
        rec['ratio'] = r0.get('kc_2_mat_theo_be_ngang')
        rec['cham'] = r0.get('dang_cham')
        rec['yaw_A'], rec['yaw_B'] = r0.get('yaw_A'), r0.get('yaw_B')
        mask_rows.append(rec)
    cv2.imwrite(f'{DIAG_DIR}/frame_{i:05d}.jpg', annotate(frm, fcs, asg, dbgi, masks, prots))

print(f'\nĐã lưu {len(keep)} frame vào {DIAG_DIR}/')
print('  vùng tô màu = mask THẬT sẽ được dán | viền trắng = vùng mũi/môi (prot)')
print('  khung đậm = bbox | đường mảnh = ô paste-back mặc định của inswapper')

if mask_rows:
    print('\n' + '=' * 70)
    print('VÙNG MŨI/MÔI CÓ ĐƯỢC NHẬN RA KHÔNG?')
    print('=' * 70)
    print(f'  {"frame":>6} {"kc/bề ngang":>12} {"chạm":>6} {"prot A":>8} {"prot B":>8} '
          f'{"yaw A":>7} {"yaw B":>7} {"đậm A":>7} {"đậm B":>7}')
    for r in mask_rows:
        rt = f"{r['ratio']:.2f}" if r.get('ratio') is not None else '--'
        ch = ('CÓ' if r['cham'] else 'ko') if r.get('cham') is not None else '--'
        ya = f"{r['yaw_A']:.2f}" if r.get('yaw_A') is not None else '--'
        yb = f"{r['yaw_B']:.2f}" if r.get('yaw_B') is not None else '--'
        wa = f"{yaw_weight(r['yaw_A']):.2f}" if r.get('yaw_A') is not None else '--'
        wb = f"{yaw_weight(r['yaw_B']):.2f}" if r.get('yaw_B') is not None else '--'
        print(f'  {r["frame"]:>6} {rt:>12} {ch:>6} {r["prot_A"]:>8.0f} {r["prot_B"]:>8.0f} '
              f'{ya:>7} {yb:>7} {wa:>7} {wb:>7}')

    zero = sum(1 for r in mask_rows if r['prot_A'] == 0 or r['prot_B'] == 0)
    print()
    if zero == len(mask_rows):
        print('  -> prot = 0 ở MỌI frame: parsenet không nhận ra mũi/môi ở góc mặt này.')
        print('     Dù ngưỡng chạm có kích hoạt cũng KHÔNG có gì để trừ ra.')
        print(f'     Nới PART_MAX_DIST (hiện {PART_MAX_DIST}) hoặc kiểm tra parsenet '
              f'có thật sự chạy không.')
    elif zero:
        print(f'  -> {zero}/{len(mask_rows)} frame có một bên prot = 0: nhận ra không ổn định.')
    else:
        print('  -> prot > 0 ở mọi frame: mũi/môi được nhận ra bình thường.')
        print('     Nếu chúng vẫn bị swap thì nguyên nhân là ngưỡng chạm, xem mục trên.')

### 6.2b Đọc kết quả chẩn đoán

Tách riêng khỏi cell quét để phân tích lại mà không phải quét video lần nữa.

In [ ]:
import numpy as np

assert rows, 'Chưa có số liệu — chạy cell quét ở 6.2 trước.'

def col(name):
    return [r.get(name) for r in rows]

def vals(name):
    return [v for v in col(name) if v is not None]

n = len(rows)
print(f'Tổng số frame đã đo: {n}\n')

# ---- 1. Detector thấy mấy mặt mỗi frame ----
print('1) Số mặt detect được mỗi frame')
from collections import Counter
for k, v in sorted(Counter(col('n_faces')).items()):
    print(f'   {k} mặt : {v:>5} frame ({v/n:>5.1%})')

missing = [r for r in rows if r['n_faces'] > 0 and not (r['co_A'] and r['co_B'])]
print(f'\n   Frame có mặt nhưng THIẾU nhãn A hoặc B: {len(missing)} ({len(missing)/n:.1%})')

# ---- 2. Tracking có mơ hồ không? ----
print('\n2) Biên an toàn của tracking (margin = sim với ref của mình − sim với ref người kia)')
print('   margin cao = chắc chắn đúng người. margin ~0 hoặc âm = tung đồng xu.')
amb_low = amb_neg = 0
for label in ('A', 'B'):
    m = vals(f'margin_{label}')
    if not m:
        print(f'   {label}: không có dữ liệu')
        continue
    m = np.array(m)
    lo, neg = int((m < 0.10).sum()), int((m < 0).sum())
    amb_low += lo
    amb_neg += neg
    print(f'   {label}: min={m.min():+.3f}  p5={np.percentile(m,5):+.3f}  '
          f'trung vị={np.median(m):+.3f}  |  <0.10: {lo} frame  |  <0 (SAI HẲN): {neg} frame')

# ---- 3. Khi chỉ detect được 1 mặt thì nó thuộc về ai? ----
one = [r for r in rows if r['n_faces'] == 1]
print(f'\n3) Frame chỉ detect được 1 mặt: {len(one)}')
if one:
    c = Counter(r.get('mot_mat_gan_cho') for r in one)
    for k, v in c.items():
        print(f'   gán cho {k}: {v} frame')
    sa = [r['mot_mat_simA'] for r in one if r.get('mot_mat_simA') is not None]
    sb = [r['mot_mat_simB'] for r in one if r.get('mot_mat_simB') is not None]
    if sa and sb:
        print(f'   sim với ref A: trung vị {np.median(sa):+.3f}   '
              f'sim với ref B: trung vị {np.median(sb):+.3f}')

# ---- 4. Vùng paste-back có tràn sang mặt người kia không? ----
print('\n4) Chồng lấn vùng paste-back (chỉ tính frame có đủ cả 2 người)')
ovBA, ovAB = vals('B_phu_len_mat_A'), vals('A_phu_len_mat_B')
big = 0
if ovBA:
    a, b = np.array(ovBA), np.array(ovAB)
    big = int((np.maximum(a, b) > 0.30).sum())
    print(f'   ô paste của B phủ lên mặt A: trung vị {np.median(a):.1%}  tối đa {a.max():.1%}')
    print(f'   ô paste của A phủ lên mặt B: trung vị {np.median(b):.1%}  tối đa {b.max():.1%}')
    print(f'   frame có chồng lấn > 30%: {big} / {len(a)} ({big/len(a):.1%})')
    worst = sorted((r for r in rows if r.get('B_phu_len_mat_A') is not None),
                   key=lambda r: -max(r['B_phu_len_mat_A'], r['A_phu_len_mat_B']))[:8]
    print('\n   8 frame chồng lấn nặng nhất:')
    print(f'   {"frame":>7} {"B phủ A":>9} {"A phủ B":>9} {"kc/bề ngang":>12} '
          f'{"margin A":>9} {"margin B":>9}')
    for r in worst:
        ma = f"{r['margin_A']:+.2f}" if r.get('margin_A') is not None else '   --'
        mb = f"{r['margin_B']:+.2f}" if r.get('margin_B') is not None else '   --'
        print(f'   {r["frame"]:>7} {r["B_phu_len_mat_A"]:>8.1%} {r["A_phu_len_mat_B"]:>8.1%} '
              f'{r["kc_2_mat_theo_be_ngang"]:>12.2f} {ma:>9} {mb:>9}')
else:
    print('   (không có frame nào detect đủ 2 người)')

# ---- 5. Kết luận ----
print('\n' + '=' * 68)
print('KẾT LUẬN')
print('=' * 68)
if big and big / max(len(ovBA), 1) > 0.02:
    print(f'- TRÀN PASTE-BACK: {big} frame có ô dán phủ >30% mặt người kia.')
    print('  Vì vòng lặp luôn dán A trước rồi B sau, ở vùng chồng lấn B LUÔN đè lên A')
    print('  -> mặt người A hiện ra mặt của nguồn B. Khớp với triệu chứng một chiều.')
else:
    print('- Không thấy tràn paste-back đáng kể.')
if amb_neg:
    print(f'- TRACKING SAI HẲN: {amb_neg} frame có margin ÂM — mặt bị gán cho nhầm người.')
elif amb_low > max(3, 0.05 * n):
    print(f'- TRACKING MƠ HỒ: {amb_low} frame có margin < 0.10, ranh giới A/B mong manh.')
else:
    print(f'- Tracking có biên an toàn rõ ràng (chỉ {amb_low} frame margin < 0.10)'
          ' -> không phải nguyên nhân.')
if missing:
    print(f'- DETECTOR RỚT MẶT: {len(missing)} frame có mặt nhưng thiếu nhãn'
          ' -> người đó giữ nguyên mặt gốc ở frame đó.')
print('=' * 68)

# ---- 6. Biểu đồ theo thời gian ----
try:
    import matplotlib.pyplot as plt
    f = col('frame')
    fig, ax = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
    ax[0].step(f, col('n_faces'), where='post', lw=1)
    ax[0].set_ylabel('số mặt'); ax[0].set_yticks([0, 1, 2, 3]); ax[0].grid(alpha=.3)
    for label, c in (('A', 'tab:orange'), ('B', 'tab:blue')):
        ax[1].plot(f, col(f'margin_{label}'), lw=1, color=c, label=f'margin {label}')
    ax[1].axhline(0, color='r', lw=1, ls='--')
    ax[1].axhline(0.15, color='orange', lw=1, ls=':')
    ax[1].set_ylabel('margin'); ax[1].legend(loc='upper right', fontsize=8); ax[1].grid(alpha=.3)
    ax[2].plot(f, [None if v is None else v * 100 for v in col('B_phu_len_mat_A')],
               lw=1, label='B phủ lên mặt A')
    ax[2].plot(f, [None if v is None else v * 100 for v in col('A_phu_len_mat_B')],
               lw=1, label='A phủ lên mặt B')
    ax[2].axhline(30, color='r', lw=1, ls='--')
    ax[2].set_ylabel('% chồng lấn'); ax[2].set_xlabel('frame')
    ax[2].legend(loc='upper right', fontsize=8); ax[2].grid(alpha=.3)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f'(bỏ qua biểu đồ: {type(e).__name__}: {e})')

### 6.3 Chạy xử lý toàn bộ video

Cell dưới dùng chính `FaceTracker` ở 6.1.

In [ ]:
import os, sys, subprocess
import numpy as np
import cv2
from tqdm import tqdm

GFPGAN_PAD = 0.4         # nới bbox bao nhiêu lần khi crop để restore
final_output = '/content/output_final.mp4'

cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps != fps or fps <= 0:       # 0.0 hoặc NaN với một số container
    print('Không đọc được fps từ video, mặc định 25.')
    fps = 25.0

# CAP_PROP_FRAME_COUNT chỉ là ước lượng (sai với VFR / mp4 thiếu index) -> chỉ dùng cho progress bar.
# Vòng lặp đọc tới khi hết frame thật sự, thay vì range(total_frames) (đếm thiếu = cụt đuôi video).
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None

# Lấy kích thước từ frame THẬT, không từ CAP_PROP_FRAME_WIDTH/HEIGHT: video quay dọc có metadata
# rotation làm hai giá trị đó bị hoán đổi so với frame OpenCV trả về -> ffmpeg nhận rawvideo sai size.
ret, frame = cap.read()
assert ret, 'Không đọc được frame nào từ video.'
height, width = frame.shape[:2]

print(f'Video: {width}x{height} @ {fps:.2f}fps, ~{total_frames} frames')

# Ghi frame thô thẳng vào ffmpeg và ghép audio ngay trong cùng một pass.
ffmpeg_cmd = [
    'ffmpeg', '-y', '-loglevel', 'error',
    '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{width}x{height}', '-r', f'{fps}', '-i', 'pipe:0',
    '-i', source_video_path,
    '-map', '0:v:0', '-map', '1:a:0?',        # '?' = không có audio thì bỏ qua, không lỗi
    '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
    '-pix_fmt', 'yuv420p',                    # để trình duyệt/IPython.display.Video phát được
    '-c:a', 'aac', '-shortest',
    final_output,
]
proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)


def enhance_face_region(img, face, pad=GFPGAN_PAD):
    """Chỉ restore vùng quanh khuôn mặt vừa swap, KHÔNG chạy trên cả frame.

    Gọi restorer.enhance() trên nguyên frame khiến GFPGAN chạy lại một bộ face detector
    thứ hai trên toàn khung (trùng lặp với app.get() vừa chạy), restore luôn cả những mặt
    trong nền không hề bị swap, và resize LANCZOS toàn frame mỗi lượt.
    """
    h, w = img.shape[:2]
    x1, y1, x2, y2 = face.bbox
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw)); x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh)); y2 = min(h, int(y2 + pad * bh))
    cw, ch = x2 - x1, y2 - y1
    if cw < 64 or ch < 64:
        return img

    crop = img[y1:y2, x1:x2]
    _, _, restored = restorer.enhance(crop, has_aligned=False,
                                      only_center_face=True, paste_back=True)
    if restored is None:
        return img
    if restored.shape[:2] != (ch, cw):
        restored = cv2.resize(restored, (cw, ch), interpolation=cv2.INTER_LANCZOS4)

    # Blend viền mềm để vùng crop không để lại đường viền hình chữ nhật thấy rõ.
    mask = np.zeros((ch, cw), np.float32)
    cv2.rectangle(mask, (int(0.12 * cw), int(0.12 * ch)),
                  (int(0.88 * cw), int(0.88 * ch)), 1.0, -1)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(2.0, 0.05 * max(cw, ch)))[..., None]
    img[y1:y2, x1:x2] = (restored * mask + crop * (1.0 - mask)).astype(np.uint8)
    return img


tracker = FaceTracker(width, height)          # định nghĩa ở cell 6.1
source_face = {'A': source_face_a, 'B': source_face_b}

pbar = tqdm(total=total_frames, unit='frame')
frames_written = 0
rc = None
try:
    while frame is not None:
        target_faces = app.get(frame)
        result_frame = frame

        if target_faces:
            assigned, dbg = tracker.step(target_faces)
            for label in dbg['late_init']:
                print(f'  [frame {frames_written}] khởi tạo danh tính {label}')

            # Ghép tất cả mặt cùng lúc TỪ FRAME GỐC (xem cell 6.1b). Gọi swapper tuần tự
            # trên result_frame làm người dán sau đè lên người dán trước, đồng thời khiến
            # swap thứ hai cắt crop từ vùng đã bị swap thứ nhất ghi đè.
            result_frame = swap_all_faces(frame, assigned, target_faces, swapper, source_face)

            if restorer is not None:
                for idx in assigned.values():
                    result_frame = enhance_face_region(result_frame, target_faces[idx])

        proc.stdin.write(np.ascontiguousarray(result_frame).tobytes())
        frames_written += 1
        pbar.update(1)

        ret, frame = cap.read()
        if not ret:
            frame = None
finally:
    # Không release trong finally thì khi bấm Stop giữa chừng, ffmpeg treo và file mp4 hỏng.
    pbar.close()
    cap.release()
    try:
        proc.stdin.close()
    except BrokenPipeError:
        pass
    rc = proc.wait()

assert rc == 0, f'ffmpeg thất bại (exit code {rc}) — xem log lỗi ở trên.'
print(f'Đã swap {frames_written} frame và ghép audio xong -> {final_output}')

## 7. Kiểm tra kết quả

Audio đã được ghép ngay trong cell trên (ffmpeg nhận frame qua pipe và mux luôn audio gốc trong cùng một pass), nên ở đây chỉ cần xác nhận file xuất ra hợp lệ.

In [ ]:
import os

assert os.path.exists(final_output) and os.path.getsize(final_output) > 0, \
    'Không tạo được video output — chạy lại cell xử lý video ở mục 6.'
print(f'{final_output}  —  {os.path.getsize(final_output) / 1e6:.1f} MB\n')

# Xác nhận có stream video (và audio, nếu video gốc có audio)
!ffprobe -v error -show_entries stream=index,codec_type,codec_name,width,height,r_frame_rate,duration \
  -of default=noprint_wrappers=1 {final_output}

## 8. Xem kết quả

In [ ]:
import os
from IPython.display import Video, display

size_mb = os.path.getsize(final_output) / 1e6
if size_mb > 50:
    # embed=True nhét toàn bộ file dưới dạng base64 vào output của notebook -> file .ipynb phình to
    # và trình duyệt dễ treo với video dài. Trường hợp đó thì tải về xem thay vì preview inline.
    print(f'Video {size_mb:.1f} MB — quá lớn để nhúng inline, chạy cell dưới để tải về máy.')
else:
    display(Video(final_output, embed=True, width=480))

In [ ]:
# Tải file về máy
from google.colab import files
files.download(final_output)

## Ghi chú / Hướng cải thiện tiếp theo

- **Flicker giữa các frame**: face swap từng frame độc lập nên đôi khi có giật/nhòe nhẹ theo thời gian. Có thể cải thiện bằng cách thêm temporal smoothing (trung bình landmark giữa các frame liền kề) hoặc dùng model chuyên video như **SimSwap** với chế độ video.
- **Tracking 2 người**: pipeline dùng khớp theo embedding ArcFace (danh tính) + vị trí làm yếu tố phụ, giải bài toán gán tối ưu toàn cục mỗi frame. Cách này chịu được occlusion dài, đổi chỗ nhanh và cắt cảnh — những thứ mà tracking thuần theo vị trí hay gán nhầm. Nếu vẫn gặp nhầm lẫn, chỉnh `MIN_SIM` (tăng để khắt khe hơn) và `POS_W` ở cell mục 6.
- **Ai là A, ai là B**: quy ước lấy ở frame đầu tiên detect được mặt — A = mặt bên trái khung hình, B = mặt bên phải. Nếu bị ngược, chỉ cần đảo 2 ảnh upload ở bước 3.
- **Tốc độ**: xử lý frame-by-frame trên Colab free (T4) sẽ khá chậm với video dài. Nên test với video ngắn (5–10s) trước. GFPGAN chỉ chạy trên vùng crop quanh mặt đã swap (không phải cả frame) nên đã nhanh hơn đáng kể; tắt hẳn bằng `USE_GFPGAN = False` nếu vẫn quá chậm.
- **inswapper_128 model**: link tải có thể thay đổi do các vấn đề về chính sách/gỡ bỏ. Cell mục 2 đã tự kiểm tra dung lượng file và báo lỗi ngay nếu tải hỏng, thay vì để tới lúc load model mới nổ.
- **Chất lượng ảnh mặt nguồn**: ảnh càng rõ, chính diện, ánh sáng đều thì kết quả swap càng tự nhiên.